In [1]:
from ultralytics import YOLO

model = YOLO("yolov8l.pt")
model.export(format="onnx", opset=12)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/jessnou/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.9 🚀 Python-3.11.9 torch-2.10.0+cu128 CPU (AMD Ryzen 5 5500)
YOLOv8l summary (fused): 112 layers, 43,668,288 parameters, 0 gradients, 165.2 GFLOPs

PyTorch: starting from 'yolov8l.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 84, 8400) (83.7 MB)
requirements: Ultralytics requirement ['onnxslim>=0.1.71'] not found, attempting AutoUpdate...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/167.3 kB ? eta -:--:--
   ━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/167.3 kB 1.4 MB/s eta 0:00:01
   ━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/167.3 kB 532.3 kB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━ 112.6/167.3 kB 1.1 MB/s eta 0

'yolov8l.onnx'

In [1]:
import cv2
import os
import json
PATH = "dat/train"

# img_path = os.path.join(PATH, "img/acdee54a-578cc9f8.jpg")
# ann_path = os.path.join(PATH, "ann/acdee54a-578cc9f8.jpg.json")

img_path = os.path.join(PATH, "img/b03ce71b-6afe07b4.jpg")
ann_path = os.path.join(PATH, "ann/b03ce71b-6afe07b4.jpg.json")

# b03ce71b-6afe07b4

def draw_polyline(img, points, color=(0,255,0), thickness=2):
    for i in range(len(points) - 1):
        p1 = tuple(map(int, points[i]))
        p2 = tuple(map(int, points[i+1]))
        cv2.line(img, p1, p2, color, thickness)


# загрузка
img = cv2.imread(img_path)
with open(ann_path, "r") as f:
    ann = json.load(f)

# рисуем только lane
for obj in ann["objects"]:
    if obj["classTitle"] == "lane":
        pts = obj["points"]["exterior"]
        if len(pts) >= 2:
            draw_polyline(img, pts)
        else:
            p1 = tuple(map(int, pts[0]))
            p2 = tuple(map(int, pts[1]))

            cv2.line(img, p1, p2, (0, 255, 0), 2)

cv2.imshow("Lanes", img)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [2]:
from LDW import LDWDetector

ldw = LDWDetector(1280, 720)
lane_polylines = []

for obj in ann["objects"]:
    if obj["classTitle"] == "lane":
        pts = obj["points"]["exterior"]
        if len(pts) >= 2:
            lane_polylines.append(pts)

ldw = LDWDetector(img.shape[1], img.shape[0], threshold_px=50)

result = ldw.process_lanes(lane_polylines)
vis = ldw.draw(img, result)

In [3]:

cv2.imshow("LDW", vis)
cv2.waitKey(0)

-1